# Qwen3.5-2B × llama.cpp: an executable architecture lab

**Goal:** inspect a real Qwen3.5-2B GGUF, identify its hybrid Gated DeltaNet and full-attention layers, run text inference, and map tensors to llama.cpp source. Start with the text model. Image input also needs a separate multimodal projector, which this lab does not use.

Set `MODEL_PATH` to a downloaded GGUF and `LLAMA_CLI` to a built executable. Install `gguf` in this notebook's kernel with `%pip install gguf`. No model is downloaded automatically.

Official model: [Qwen/Qwen3.5-2B](https://huggingface.co/Qwen/Qwen3.5-2B). Example GGUF: [SoAIHQ/Qwen3.5-2B-GGUF](https://huggingface.co/SoAIHQ/Qwen3.5-2B-GGUF). Execution graph: [llama.cpp qwen35.cpp](https://github.com/ggml-org/llama.cpp/blob/master/src/models/qwen35.cpp).


## 1. Point this notebook at your setup

Qwen3.5-2B uses **24 layers: 18 Gated DeltaNet and 6 full-attention layers**. Do not use a Qwen3 or Qwen2.5 file for this lab. The text GGUF can be used without the optional image projector. GGUF weight types may be mixed; inspect the actual types below.

To learn where your existing file came from, inspect its `general.name` and `general.architecture`. These identify its contents, though they do not prove its download source.


In [3]:
from pathlib import Path
import importlib.util, sys
import numpy as np

MODEL_PATH = Path('models/Qwen3.5-2B-Q4_K_M.gguf').expanduser()  # EDIT ME
LLAMA_CLI = Path(shutil.which("llama-cli"))
print('Python:', sys.executable)
print('GGUF exists:', MODEL_PATH.exists(), MODEL_PATH.resolve())
print('CLI exists:', LLAMA_CLI.exists(), LLAMA_CLI.resolve())
print('gguf installed:', importlib.util.find_spec('gguf') is not None)


Python: c:\Users\adams\SiliconBadgers\AI_stuff\venv\Scripts\python.exe
GGUF exists: True C:\Users\adams\SiliconBadgers\AI_stuff\models\Qwen3.5-2B-Q4_K_M.gguf
CLI exists: True C:\Users\adams\AppData\Local\Microsoft\WinGet\Packages\ggml.llamacpp_Microsoft.Winget.Source_8wekyb3d8bbwe\llama-cli.exe
gguf installed: True


## 2. Mental model: two layer types

`text → token IDs → embedding → 24 hybrid layers → final normalization → vocabulary logits → sampler`.

Most layers use **Gated DeltaNet**, a recurrent linear-attention path with its own state. Every fourth layer uses **full attention** with a key/value (KV) cache. Both have feed-forward networks (FFNs) and residual connections. Prefill processes a prompt; decode advances one token and updates the relevant state. A GGUF holds model data; llama.cpp constructs and executes the operation graph.

Before running code, predict which tensors appear in a DeltaNet layer versus a full-attention layer. Compare `blk.0` and `blk.3` below.


In [4]:
# A small numerical version of two common operations. These are FP32 teaching examples,
# not the exact quantized ggml kernels or a full transformer layer.
def rmsnorm(x, weight, eps=1e-6):
    return x / np.sqrt(np.mean(x*x, axis=-1, keepdims=True) + eps) * weight

def silu(x):
    return x / (1 + np.exp(-x))

rng = np.random.default_rng(3)
x = rng.normal(size=(2, 4)).astype(np.float32)  # 2 tokens, hidden width 4
up = rng.normal(size=(4, 8)).astype(np.float32)
gate = rng.normal(size=(4, 8)).astype(np.float32)
down = rng.normal(size=(8, 4)).astype(np.float32)
y = x + (silu(rmsnorm(x, np.ones(4)) @ gate)
         * (rmsnorm(x, np.ones(4)) @ up)) @ down
print('input:', x.shape, 'gate/up:', up.shape, 'output:', y.shape)
print('In qwen35.cpp, locate build_norm(... RMS), build_ffn(... SILU), and the residual ggml_add.')


input: (2, 4) gate/up: (4, 8) output: (2, 4)
In qwen35.cpp, locate build_norm(... RMS), build_ffn(... SILU), and the residual ggml_add.


## 3. Read the GGUF without loading all weights into NumPy

`GGUFReader` memory maps the file. The table prints metadata keys and a small number of scalar values; it deliberately skips huge tokenizer arrays. Tensor `shape` follows GGUF/ggml dimension order, which can appear reversed relative to NumPy or PyTorch linear-layer weight shapes. **Do not infer a mathematical transpose from the printed tuple alone.**

In [11]:
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f'Edit MODEL_PATH first: {MODEL_PATH.resolve()}')
try:
    from gguf import GGUFReader
except ImportError as e:
    raise ImportError('Install gguf in this notebook kernel: %pip install gguf') from e
reader = GGUFReader(str(MODEL_PATH))
print('Metadata fields:', len(reader.fields), 'Tensors:', len(reader.tensors))

def scalar(key, default=None):
    field = reader.fields.get(key)
    if field is None or field.types[0].name == "ARRAY":
        return default
    return field.contents()

for key in reader.fields:
    if key.startswith('tokenizer.ggml.tokens') or key.startswith('tokenizer.ggml.scores'):
        continue
    value = scalar(key, '<array or structured value>')
    if key.startswith(('general.', 'qwen3.', 'tokenizer.ggml.model')):
        print(f'{key:53} {str(value)[:95]}')


Metadata fields: 50 Tensors: 335
general.architecture                                  qwen35
general.type                                          model
general.name                                          Qwen3.5 2B
general.basename                                      Qwen3.5
general.size_label                                    2B
general.license                                       apache-2.0
general.license.link                                  https://huggingface.co/Qwen/Qwen3.5-2B/blob/main/LICENSE
general.base_model.count                              1
general.base_model.0.name                             Qwen3.5 2B Base
general.base_model.0.organization                     Qwen
general.base_model.0.repo_url                         https://huggingface.co/Qwen/Qwen3.5-2B-Base
general.tags                                          <array or structured value>
tokenizer.ggml.model                                  gpt2
general.quantization_version                          2
genera

## 4. Make the model visible: tensor inventory

Look at `token_embd`, layer 0 `attn_q/k/v`, `attn_output`, `ffn_gate/up/down`, and `output`. Group by tensor type and by component. `data.nbytes` is on-disk tensor payload size, **not** runtime activation size or exact device memory use.

In [12]:
from collections import Counter, defaultdict

def shape_of(t):
    return tuple(int(v) for v in t.shape)

rows = [{'name': t.name, 'shape': shape_of(t), 'type': t.tensor_type.name,
         'bytes': int(t.data.nbytes)} for t in reader.tensors]
print('First 24 tensors:')
for r in rows[:24]:
    print(f"{r['name']:40} {str(r['shape']):22} {r['type']:12} {r['bytes']/2**20:8.2f} MiB")
print('\nType counts:', Counter(r['type'] for r in rows))
print('Total tensor payload: %.1f MiB' % (sum(r['bytes'] for r in rows)/2**20))
print('\nLayer 0 (DeltaNet) and layer 3 (full attention):')
for r in rows:
    if r['name'].startswith(('blk.0.', 'blk.3.')):
        print(f"{r['name']:40} {str(r['shape']):22} {r['type']}")


First 24 tensors:
output_norm.weight                       (2048,)                F32              0.01 MiB
token_embd.weight                        (2048, 248320)         Q6_K           397.85 MiB
blk.0.attn_gate.weight                   (2048, 2048)           Q4_K             2.25 MiB
blk.0.attn_norm.weight                   (2048,)                F32              0.01 MiB
blk.0.attn_qkv.weight                    (2048, 6144)           Q6_K             9.84 MiB
blk.0.ffn_down.weight                    (6144, 2048)           Q6_K             9.84 MiB
blk.0.ffn_gate.weight                    (2048, 6144)           Q4_K             6.75 MiB
blk.0.ffn_up.weight                      (2048, 6144)           Q4_K             6.75 MiB
blk.0.post_attention_norm.weight         (2048,)                F32              0.01 MiB
blk.0.ssm_a                              (16,)                  F32              0.00 MiB
blk.0.ssm_alpha.weight                   (2048, 16)             Q4_K             0

## 5. Shape and memory exercise for *your file*

Inspect the metadata first. In Qwen3.5-2B only **6 of 24 layers** use full attention and its conventional KV cache. The other **18 layers** have Gated DeltaNet state. The estimate below covers **only the full-attention KV tensors** for one sequence at 2 bytes per element; it does not include DeltaNet state, activations, padding, or implementation-specific buffers. Using 24 layers in this formula would be wrong.


In [ ]:
arch = scalar('general.architecture')
print('GGUF architecture:', arch)
assert arch and ('qwen35' in str(arch).lower() or 'qwen3_5' in str(arch).lower()), (
    'This is not a Qwen3.5 GGUF; inspect general.architecture and change MODEL_PATH.')
prefix = str(arch) + '.'
def get(*names):
    for n in names:
        v = scalar(prefix + n)
        if v is not None:
            return int(v)
    return None

L = get('block_count')
D = get('embedding_length')
H = get('attention.head_count')
HK = get('attention.head_count_kv')
HD = get('attention.key_length') or (D // H if D and H else None)
F = get('feed_forward_length')
print(dict(layers=L, hidden=D, query_heads=H, kv_heads=HK, head_dim=HD, ffn_width=F))
print('Inspect the first four blocks in the tensor listing: 0–2 DeltaNet, 3 full attention.')
if L == 24 and HK and HD:
    attention_layers = 6  # Official Qwen3.5-2B 3:1 layout, not all 24 layers
    for positions in (128, 1024, 4096, 8192):
        mib = 2 * attention_layers * positions * HK * HD * 2 / 2**20
        print(f'{positions:5d} positions → ~{mib:8.1f} MiB full-attention KV alone')
else:
    print('Review metadata and actual layer layout before computing KV size.')


GGUF architecture: qwen35
{'layers': 25, 'hidden': 2048, 'query_heads': 8, 'kv_heads': 2, 'head_dim': 256, 'ffn_width': 6144}
Inspect the first four blocks in the tensor listing: 0–2 DeltaNet, 3 full attention.
Review metadata and actual layer layout before computing KV size.


In [14]:
print(repr(scalar("general.architecture")))
print(repr(scalar("general.name")))

'qwen35'
'Qwen3.5 2B'


## 6. Tokenization and actual inference with llama.cpp

Run `llama-cli --help` for flags in **your build**. Below uses a short bounded run and asks llama.cpp to print timing data. In a terminal, you can also use `llama-tokenize` if built. The first load may be slower due to file/cache initialization. For Qwen3 instruct models, interactive chat uses the model's template; this plain prompt example is for learning inference timing and may yield odd completions. Avoid comparing answer quality using this raw prompt.

In [8]:
import subprocess
if not LLAMA_CLI.is_file():
    print('Set LLAMA_CLI to your built executable to run this cell.')
else:
    cmd = [str(LLAMA_CLI.resolve()), '-m', str(MODEL_PATH.resolve()),
           '-p', 'The key/value cache stores', '-n', '24', '-c', '512',
           '--temp', '0', '--perf', '--no-display-prompt', '-st']
    print('Running:', ' '.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=180)
    print('exit code:', result.returncode)
    print('stdout:\n', result.stdout[-5000:])
    print('stderr (includes loader/timing information):\n', result.stderr[-6500:])


Running: C:\Users\adams\AppData\Local\Microsoft\WinGet\Packages\ggml.llamacpp_Microsoft.Winget.Source_8wekyb3d8bbwe\llama-cli.exe -m C:\Users\adams\SiliconBadgers\AI_stuff\models\Qwen3.5-2B-Q4_K_M.gguf -p The key/value cache stores -n 24 -c 512 --temp 0 --perf --no-display-prompt -st
exit code: 0
stdout:
 

Loading model... 

â–„â–„ â–„â–„
â–ˆâ–ˆ â–ˆâ–ˆ
â–ˆâ–ˆ â–ˆâ–ˆ  â–€â–€â–ˆâ–„ â–ˆâ–ˆâ–ˆâ–„â–ˆâ–ˆâ–ˆâ–„  â–€â–€â–ˆâ–„    â–„â–ˆâ–ˆâ–ˆâ–ˆ â–ˆâ–ˆâ–ˆâ–ˆâ–„ â–ˆâ–ˆâ–ˆâ–ˆâ–„
â–ˆâ–ˆ â–ˆâ–ˆ â–„â–ˆâ–€â–ˆâ–ˆ â–ˆâ–ˆ â–ˆâ–ˆ â–ˆâ–ˆ â–„â–ˆâ–€â–ˆâ–ˆ    â–ˆâ–ˆ    â–ˆâ–ˆ â–ˆâ–ˆ â–ˆâ–ˆ â–ˆâ–ˆ
â–ˆâ–ˆ â–ˆâ–ˆ â–€â–ˆâ–„â–ˆâ–ˆ â–ˆâ–ˆ â–ˆâ–ˆ â–ˆâ–ˆ â–€â–ˆâ–„â–ˆâ–ˆ â–ˆâ–ˆ â–€â–ˆâ–ˆâ–ˆâ–ˆ â–ˆâ–ˆâ–ˆâ–ˆâ–€ â–ˆâ–ˆâ–ˆâ–ˆâ–€
                                    â–ˆâ–ˆ    â–ˆâ–ˆ
                                    â–€â–€    â–€â–€

build      : b11193-4e7481175
model      : C:\Users\adams\SiliconBadgers\AI_stuff\models\Qwen3.5-2B-Q4_K_M.gguf
ftype      : Q4_K - Medium
modalities : text

available commands:
  /exit or 

## 7. Inspect the C++ that executes Qwen3.5

Open [src/models/qwen35.cpp](https://github.com/ggml-org/llama.cpp/blob/master/src/models/qwen35.cpp) and search within your checkout for `load_arch_tensors`, `build_arch_graph`, `build_qkvz`, `build_attn`, and the DeltaNet/SSM operations. Match the tensors printed for `blk.0` and `blk.3` to the distinct branches. ggml specifies operations; CPU/GPU backends execute them. The source can evolve, so search symbols rather than assuming fixed line numbers.

**Do not read `src/models/qwen3.cpp` as this model's graph.** Qwen3.5-2B is hybrid; the ordinary Qwen3 graph is different.


## 8. Your Compute 1 hardware notes

1. Inventory the matrix multiplications and elementwise kernels for **both** a DeltaNet layer and a full-attention layer.
2. Compare the recurrent DeltaNet state with attention's KV cache. Which storage grows with sequence length?
3. Record weight quantization separately from activation and accumulator formats. GGUF type does not specify your RTL accumulator precision.
4. Run a short prompt in llama.cpp and record prompt evaluation versus generation timing.
5. Label the decisions your club must make: numeric formats, dataflow, buffering, interfaces, and which kernels your compute unit owns.
